In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

c:\Users\тема\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_dir = Path(r"C:\Users\тема\Desktop")
train_path = base_dir / "train_prepared.csv"
model_name = "DeepPavlov/rubert-base-cased"
output_dir = base_dir / f"saved_model_{model_name.split('/')[-1]}"
os.makedirs(output_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# Загрузка подготовленного датасета
df_train = pd.read_csv(train_path)
text_col = "text"
label_col = "label"

# Разделение на train/val: 80% / 20%
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train[text_col].values,
    df_train[label_col].values,
    test_size=0.20,
    random_state=42,
    stratify=df_train[label_col]
)

In [4]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=100):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_dataset = EmotionDataset(train_texts, train_labels, tokenizer)
val_dataset = EmotionDataset(val_texts, val_labels, tokenizer)

batch_size = 16
max_len = 100
epochs = 2
lr = 1e-5 

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

best_val_f1 = 0.0
best_model_state = None
emotion_names = ["joy", "sadness", "surprise", "fear", "anger"]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20796.99it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/a

In [7]:
for epoch in range(epochs):
    # Обучение
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (train)"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    # Валидация
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} (val)"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    val_f1 = f1_score(val_true, val_preds, average="macro")
    print(f"\nEpoch {epoch+1}: Val F1 (macro) = {val_f1:.4f}")

    report = classification_report(val_true, val_preds, target_names=emotion_names, digits=4)
    print("Classification report (validation):")
    print(report)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}

# Сохранение лучшей модели
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n Модель сохранена в: {output_dir}")
else:
    print("Не удалось сохранить модель")

Epoch 1/2 (val): 100%|██████████| 281/281 [04:09<00:00,  1.13it/s]



Epoch 1: Val F1 (macro) = 0.7869
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.7997    0.9535    0.8699      1977
     sadness     0.8331    0.6353    0.7209       998
    surprise     0.8081    0.7938    0.8009       451
        fear     0.8232    0.8328    0.8280       341
       anger     0.8010    0.6457    0.7151       717

    accuracy                         0.8082      4484
   macro avg     0.8130    0.7722    0.7869      4484
weighted avg     0.8100    0.8082    0.8018      4484



Epoch 2/2 (val): 100%|██████████| 281/281 [04:11<00:00,  1.12it/s]



Epoch 2: Val F1 (macro) = 0.7869
Classification report (validation):
              precision    recall  f1-score   support

         joy     0.7997    0.9535    0.8699      1977
     sadness     0.8331    0.6353    0.7209       998
    surprise     0.8081    0.7938    0.8009       451
        fear     0.8232    0.8328    0.8280       341
       anger     0.8010    0.6457    0.7151       717

    accuracy                         0.8082      4484
   macro avg     0.8130    0.7722    0.7869      4484
weighted avg     0.8100    0.8082    0.8018      4484



Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]


 Модель сохранена в: C:\Users\тема\Desktop\saved_model_rubert-base-cased
